# 01 · Temporal DeepSet — Handover Cell Selection
**Notebook location:** `notebooks/modeling/01_temporal_deepset.ipynb`  
**Project root:** `../../` (two levels up)

### Architecture
```
Input (B,10,25,4) + Mask (B,10)
  TimeDistributed(LSTM 64)          shared temporal encoder per cell
  TimeDistributed(Dense 64×2)       Φ: per-cell non-linear embedding
  MaskedGlobalAveragePooling        z = Σ Φ(hᵢ)·maskᵢ / Σ maskᵢ
  Concat [Φ(hᵢ) ‖ z] + Dense 64   ρ: cell scores in global context
  Masked Softmax (pad → −∞)         (B,10) output over candidate cells
```
### Loss Strategy
| Mode | Function | Use case |
|---|---|---|
| `focal` | Focal Loss γ=2.0 α=0.25 | Default — combats Cell-0 dominance |
| `cce` | CCE label_smoothing=0.1 | Alternative — softer targets |

## Section 1 · Environment & Paths

In [ ]:
# ─── Section 1 · Environment, Canonical Paths, GPU, MLflow ───────────────────
#
# This notebook lives at:
#   <project_root>/notebooks/modeling/temporal_deepset.ipynb
#
# We use Path("../../").resolve() to anchor ALL I/O at the project root,
# regardless of whether Jupyter was launched from:
#   • the project root          → "../../" resolves correctly
#   • notebooks/modeling/       → "../../" resolves correctly
# Never use Path(".").resolve() — that changes with the launch directory.

import os, sys, warnings, json, pickle, logging, datetime, gc, re
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing  import List, Tuple, Dict

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from sklearn.preprocessing      import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics            import (classification_report,
                                        confusion_matrix,
                                        top_k_accuracy_score)

sns.set_theme(style="whitegrid", font_scale=1.05)

# ── Project root via explicit relative anchor ─────────────────────────────────
_ROOT = Path("../../").resolve()   # notebooks/modeling/ → up two → project root

PATHS = dict(
    data       = _ROOT / "dataset" / "temporal_deepset_cache",
    models     = _ROOT / "models",
    tb_logs    = _ROOT / "tb_logs" / "deepset",
    metrics    = _ROOT / "metrics"/ "temporal_deepset",
    mlruns     = _ROOT / "mlflow"  / "mlruns",
)
for p in PATHS.values():
    os.makedirs(str(p), exist_ok=True)

# ── Dual logging: stdout + metrics/training.log ───────────────────────────────
_log_file = PATHS["metrics"] /"temporal_deepset_training.log"
_handlers = [logging.StreamHandler(sys.stdout),
             logging.FileHandler(str(_log_file), mode="w")]
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s │ %(levelname)-8s │ %(message)s",
                    datefmt="%H:%M:%S", handlers=_handlers)
log = logging.getLogger("temporal_deepset")
log.info("Project root : %s", _ROOT)
for k, v in PATHS.items():
    log.info("  %-10s → %s", k, v)

# ── GPU ───────────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices("GPU")
log.info("GPUs: %d", len(gpus))
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
if not gpus:
    log.warning("No GPU — running on CPU.")

policy = mixed_precision.Policy("mixed_float16")
mixed_precision.set_global_policy(policy)
log.info("Mixed precision: compute=%s  vars=%s",
         policy.compute_dtype, policy.variable_dtype)

# ── MLflow: EC2 via SSH tunnel (or env-var override) ─────────────────────────
# Default: tunnel forwards EC2:5000 → localhost:5000
# Override: export MLFLOW_TRACKING_URI=http://<EC2_PUBLIC_IP>:5000
# ── MLflow: EC2 via SSH tunnel (or env-var override) ─────────────────────────
_MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")

try:
    import mlflow
    import mlflow.tensorflow
    import requests

    mlflow.set_tracking_uri(_MLFLOW_URI)

    # Close dangling notebook runs safely
    try:
        active = mlflow.active_run()
        if active is not None:
            log.warning("Closing dangling MLflow run: %s", active.info.run_id)
            mlflow.end_run()
    except Exception as e:
        log.warning("Could not close previous MLflow run: %s", e)

    # Check server health
    try:
        r = requests.get(f"{_MLFLOW_URI}/health", timeout=3)
        MLFLOW_OK = r.status_code == 200
    except Exception:
        MLFLOW_OK = False

    if MLFLOW_OK:
        mlflow.set_experiment("Temporal_deepset")
        log.info("MLflow → %s", _MLFLOW_URI)
    else:
        log.warning(
            "MLflow server unreachable at %s — is SSH tunnel open?",
            _MLFLOW_URI
        )

except ImportError:
    MLFLOW_OK = False
    log.warning("mlflow not installed.")

2026-05-30 10:27:33.929887: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-30 10:27:33.929956: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-30 10:27:33.947383: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

## Section 2 · Hyperparameters

In [ ]:
# ─── Section 2 · Hyperparameters ─────────────────────────────────────────────
HP = dict(
    # ── Data ──────────────────────────────────────────────────────────────────
    MAX_CELLS    = 10,
    OBS_STEPS    = 25,
    N_FEATS      = 3,

    # ── Loss toggle ───────────────────────────────────────────────────────────
    # "focal"  → Focal Loss (γ=2.0, α=0.25) — handles Cell-0 dominance
    # "cce"    → CategoricalCrossentropy(label_smoothing=0.1)
    LOSS_TYPE    = "focal",
    FOCAL_GAMMA  = 2.0,
    FOCAL_ALPHA  = 0.25,
    LABEL_SMOOTH = 0.1,
    PHI_LAYERS  = 2,

    # ── Architecture ──────────────────────────────────────────────────────────
    LSTM_UNITS   = 64,
    PHI_DIM      = 64,
    DROPOUT      = 0.25,

    # ── Training ──────────────────────────────────────────────────────────────
    BATCH_SIZE   = 128,
    EPOCHS       = 60,
    LR_INIT      = 1e-3,
    LR_WARMUP_EP = 4,
    LR_DECAY_EP  = 20,
    )
ALL_LABELS = list(range(HP["MAX_CELLS"]))   # [0..9] — passed to top_k_accuracy_score
log.info("HP loaded — LOSS_TYPE='%s'", HP["LOSS_TYPE"])

08:31:42 │ INFO     │ HP loaded — LOSS_TYPE='focal'


## Section 3 · Loss Functions

In [ ]:
# ─── Section 3 · Loss Functions ───────────────────────────────────────────────
#
# WHY FOCAL LOSS?
# Cell 0 accounts for ~53% of all windows. Standard CCE finds it trivially
# easy to predict Cell 0 always and still achieve ~53% accuracy.
# Focal Loss down-weights well-classified examples via (1-p_t)^γ,
# forcing the model to focus on the hard minority cells (3-7).
#
# LABELS ARE ONE-HOT (depth=MAX_CELLS=10) for both loss functions.
# This is required because:
#  • Focal loss needs per-class probabilities aligned with a dense target
#  • CCE with label_smoothing only works on dense targets in Keras
# For evaluation (sklearn), we convert back to integer via argmax.

def focal_loss(gamma: float = 2.0, alpha: float = 0.25):
    """
    Multi-class Focal Loss.
        FL = -alpha * (1 - p_t)^gamma * log(p_t)
    where p_t = y_true * y_pred  (element-wise, then summed).

    Args:
        gamma: focusing parameter — higher = more focus on hard examples
        alpha: class-balance weight (scalar, applied uniformly here;
               per-class alpha via class_weight in model.fit)
    """
    def _loss(y_true, y_pred):
        y_pred  = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)
        y_true  = tf.cast(y_true, tf.float32)
        ce      = -y_true * tf.math.log(y_pred)           # (B, C) per-class CE
        p_t     = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)  # (B,1)
        focal_w = alpha * tf.pow(1.0 - p_t, gamma)        # (B,1) weight
        return tf.reduce_mean(tf.reduce_sum(focal_w * ce, axis=-1))
    _loss.__name__ = f"focal_g{gamma}_a{alpha}"
    return _loss


def build_loss(hp: dict):
    """Return the configured loss function and a display name."""
    if hp["LOSS_TYPE"] == "focal":
        return focal_loss(hp["FOCAL_GAMMA"], hp["FOCAL_ALPHA"]), "FocalLoss"
    else:
        return (tf.keras.losses.CategoricalCrossentropy(
                    label_smoothing=hp["LABEL_SMOOTH"]),
                f"CCE(smooth={hp['LABEL_SMOOTH']})")


LOSS_FN, LOSS_NAME = build_loss(HP)
log.info("Loss function : %s", LOSS_NAME)

08:31:42 │ INFO     │ Loss function : FocalLoss


## Section 4 · Data Pipeline (One-Hot Labels)

In [ ]:
# ─── SECTION 4: UNIFIED DATA PIPELINE (Strictly Balanced) ───────────────────
#
# This cell handles everything: Loading, Shuffling, Scaling, and tf.data Creation.
# It ensures that Train, Val, and Test are ALL perfectly balanced.
SEED = 42
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path
import logging
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import re

try: _r = _ROOT
except NameError: _r = Path("../../").resolve()
try: log.info
except NameError: log = logging.getLogger("dummy"); log.setLevel(logging.INFO)

def parse_nb_array(s, max_len=10, fill=0.0):
    if pd.isna(s): return [fill] * max_len
    cleaned = re.sub(r'[\[\]]', '', str(s)).strip()
    parts = re.split(r'[,;]', cleaned)
    vals = []
    for p in parts[:max_len]:
        p = p.strip()
        if p == '' or p.lower() in ('nan', 'none'): vals.append(fill)
        else:
            try: vals.append(float(p))
            except: vals.append(fill)
    vals += [fill] * (max_len - len(vals))
    return vals

def parse_nb_ids(s, max_len=10):
    if pd.isna(s): return [0] * max_len
    cleaned = re.sub(r'[\[\]]', '', str(s)).strip()
    parts = re.split(r'[,;]', cleaned)
    ids = []
    for p in parts[:max_len]:
        p = p.strip()
        try: ids.append(int(float(p)))
        except: ids.append(0)
    ids += [0] * (max_len - len(ids))
    return ids

def load_and_create_datasets(root_dir):
    raw_path = root_dir / "dataset" / "raw" / "handover_dataset.csv"
    log.info(f"Loading raw data from {raw_path}...")
    df = pd.read_csv(raw_path, low_memory=False)
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed")
    
    df["nb_ids"]   = df["nb_cell_ids"].apply(parse_nb_ids)
    df["nb_rsrps"] = df["nb_rsrps"].apply(parse_nb_array)
    df["nb_sinrs"] = df["nb_sinrs"].apply(parse_nb_array)
    df["nb_loads"] = df["nb_loads"].apply(parse_nb_array)
    
    df.sort_values(["ue_id", "timestamp"], inplace=True)
    
    all_X, all_M, all_y, groups = [], [], [], []
    T_win, K_cells = HP["OBS_STEPS"], HP["MAX_CELLS"]
    
    # Use local RNG to ensure every sample is shuffled
    rng_shuf = np.random.default_rng(SEED)
    
    log.info("Building sequences with 3 features and mandatory shuffling...")
    for ue_id, grp in df.groupby("ue_id", sort=False):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        if len(grp) < T_win + 1: continue
        
        n_rows = len(grp)
        cell_feat = np.zeros((n_rows, K_cells, 3), dtype=np.float32)
        for i in range(n_rows):
            rs, sn, ld = grp.at[i, "nb_rsrps"], grp.at[i, "nb_sinrs"], grp.at[i, "nb_loads"]
            for k in range(K_cells):
                cell_feat[i, k, 0], cell_feat[i, k, 1], cell_feat[i, k, 2] = rs[k], sn[k], ld[k]
                
        opt_ids = grp["optimal_cell_id"].values
        ue_nb_ids = grp["nb_ids"].values
        ue_srv_ids = grp["serving_cell_id"].values
        
        for t in range(T_win, n_rows):
            X_w = cell_feat[t-T_win : t] 
            p = rng_shuf.permutation(K_cells)
            X_w_shuf = X_w[:, p, :].transpose(1, 0, 2)
            M_w = (X_w_shuf[:, -1, 0] != 0.0).astype(np.float32)
            
            opt_id = int(opt_ids[t])
            current_nb_ids = ue_nb_ids[t]
            try:
                orig_idx = current_nb_ids.index(opt_id)
                new_y = np.where(p == orig_idx)[0][0]
            except:
                srv_id = int(ue_srv_ids[t])
                try:
                    orig_srv = current_nb_ids.index(srv_id)
                    new_y = np.where(p == orig_srv)[0][0]
                except: new_y = 0 
            
            all_X.append(X_w_shuf); all_M.append(M_w); all_y.append(new_y); groups.append(ue_id)

    X, M, y, groups = np.array(all_X), np.array(all_M), np.array(all_y), np.array(groups)
    
    # Split
    ue_list = np.unique(groups)
    np.random.default_rng(SEED).shuffle(ue_list)
    n_te, n_va = int(len(ue_list)*0.15), int(len(ue_list)*0.15)
    ue_te, ue_va = set(ue_list[:n_te]), set(ue_list[n_te:n_te+n_va])
    
    idx_tr = np.where([u not in ue_te and u not in ue_va for u in groups])[0]
    idx_va = np.where([u in ue_va for u in groups])[0]
    idx_te = np.where([u in ue_te for u in groups])[0]
    
    # Scale (Optimized Vectorized Scaling)
    scaler = StandardScaler()
    
    # 1. Fit the scaler on the valid cells of the training set
    tr_mask = M[idx_tr] == 1.0
    if np.any(tr_mask):
        # Extract valid training cells and reshape to (Samples, Features)
        scaler.fit(X[idx_tr][tr_mask].reshape(-1, 3))
    
    X_n = X.copy()
    full_mask = M == 1.0
    
    # 2. Transform the entire dataset's valid cells
    if np.any(full_mask):
        # Extract all valid cells across the entire array
        # This yields an array of shape (Num_Valid_Cells, T_win, 3)
        valid_cells = X_n[full_mask]
        
        # Flatten to 2D for transformation, then immediately reshape back to 3D
        transformed_cells = scaler.transform(valid_cells.reshape(-1, 3)).reshape(valid_cells.shape)
        
        # Map the scaled values exactly back to their original positions in the 4D array
        X_n[full_mask] = transformed_cells
    
    # Final Balanced Check
    for lbl, arr in [("Train", y[idx_tr]), ("Val", y[idx_va]), ("Test", y[idx_te])]:
        u, c = np.unique(arr, return_counts=True)
        dist = dict(zip(u.tolist(), c.tolist()))
        avg = len(arr)/K_cells
        is_bal = all(abs(v-avg) < 0.1*avg for v in dist.values())
        log.info(f"{lbl} balanced: {'✅' if is_bal else '❌'} | {dist}")
        if not is_bal: raise ValueError(f"CRITICAL: {lbl} is not balanced!")

    # Return split data
    return (X_n[idx_tr], M[idx_tr], y[idx_tr],
            X_n[idx_va], M[idx_va], y[idx_va],
            X_n[idx_te], M[idx_te], y[idx_te], scaler)

# Run pipeline
X_tr, M_tr, y_tr, X_va, M_va, y_va, X_te, M_te, y_te, scaler = load_and_create_datasets(_ROOT)

# ─── Dataset Creation ─────────────────────────────────────────────────────────

def make_ds(X, M, y, shuffle=False):
    y_oh = tf.one_hot(y, depth=HP["MAX_CELLS"]).numpy().astype(np.float32)
    if len(M.shape) == 2: M = M[..., np.newaxis]
    with tf.device('/CPU:0'):
        ds = tf.data.Dataset.from_tensor_slices(({"cells": X, "mask": M}, y_oh))
    if shuffle: ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(HP["BATCH_SIZE"]).prefetch(tf.data.AUTOTUNE)

ds_tr = make_ds(X_tr, M_tr, y_tr, shuffle=True)
ds_va = make_ds(X_va, M_va, y_va)
ds_te = make_ds(X_te, M_te, y_te)

steps_per_epoch = int(np.ceil(len(X_tr) / HP["BATCH_SIZE"]))
cw_vals = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
CLASS_WEIGHT = {int(c): float(w) for c, w in enumerate(cw_vals)}

log.info("Datasets ready. Train batches: %d", len(ds_tr))

08:31:42 │ INFO     │ Loading raw data from /home/wassimmchichi/Downloads/Handover_projects/dataset/raw/handover_dataset.csv...


08:31:48 │ INFO     │ Building sequences with 3 features and mandatory shuffling...
08:31:55 │ INFO     │ Train balanced: ✅ | {0: 5863, 1: 5764, 2: 5928, 3: 5763, 4: 5918, 5: 5835, 6: 5768, 7: 5760, 8: 5751, 9: 5610}
08:31:55 │ INFO     │ Val balanced: ✅ | {0: 1259, 1: 1201, 2: 1238, 3: 1256, 4: 1230, 5: 1268, 6: 1203, 7: 1246, 8: 1297, 9: 1222}
08:31:55 │ INFO     │ Test balanced: ✅ | {0: 1252, 1: 1231, 2: 1261, 3: 1204, 4: 1252, 5: 1305, 6: 1288, 7: 1255, 8: 1238, 9: 1134}
08:31:56 │ INFO     │ Datasets ready. Train batches: 453


## Section 5 · Custom Layer — MaskedGlobalAveragePooling

In [ ]:
# ─── Section 5 · Custom Keras Layer ──────────────────────────────────────────

class MaskedGlobalAveragePooling(keras.layers.Layer):
    """
    Correct DeepSet aggregation: z = Σ Φ(hᵢ)·maskᵢ / Σ maskᵢ

    Standard GlobalAveragePooling1D divides by MAX_CELLS=10 even when
    only 7 real cells are present, diluting the context vector by 30%.
    This layer uses the actual count of real cells as the denominator.
    """
    def call(self, phi, mask):
        # phi : (B, C, D)   mask : (B, C, 1)
        summed = tf.reduce_sum(phi * mask, axis=1)
        count  = tf.maximum(tf.reduce_sum(mask, axis=1), 1e-8)
        return summed / count

    def get_config(self): return super().get_config()

print("MaskedGlobalAveragePooling defined.")

MaskedGlobalAveragePooling defined.


## Section 6· Learning Schedule

In [ ]:
# ─── Learning-Rate Schedule ───────────────────────────────────────────────────

class WarmUpCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    """Linear warm-up → cosine decay → flat at lr_min. Step-based (not epoch)."""
    def __init__(self, lr_max, lr_min, warmup_steps, decay_steps):
        super().__init__()
        self.lr_max=float(lr_max); self.lr_min=float(lr_min)
        self.warmup_steps=float(warmup_steps); self.decay_steps=float(decay_steps)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        warmup = self.lr_max * step / tf.maximum(self.warmup_steps, 1.0)
        cosine = self.lr_min + 0.5*(self.lr_max-self.lr_min)*(
            1.0 + tf.cos(np.pi *
                tf.minimum(step-self.warmup_steps, self.decay_steps)
                / self.decay_steps))
        return tf.where(step < self.warmup_steps, warmup, cosine)

    def get_config(self):
        return {"lr_max":self.lr_max,"lr_min":self.lr_min,
                "warmup_steps":self.warmup_steps,"decay_steps":self.decay_steps}

warmup_steps = HP["LR_WARMUP_EP"] * steps_per_epoch
decay_steps  = HP["LR_DECAY_EP"]  * steps_per_epoch
lr_sched = WarmUpCosineDecay(HP["LR_INIT"], HP["LR_INIT"]*0.01,
                              warmup_steps, decay_steps)

# Plot → metrics/
ep_lr = [float(lr_sched(e*steps_per_epoch)) for e in range(HP["EPOCHS"])]
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ep_lr, lw=2, color="#2196F3")
ax.axvline(HP["LR_WARMUP_EP"], color="orange", ls="--", lw=1.2,
           label=f"Warm-up end (ep {HP['LR_WARMUP_EP']})")
ax.axvline(HP["LR_WARMUP_EP"]+HP["LR_DECAY_EP"], color="red",
           ls="--", lw=1.2, label="Cosine end")
ax.set(xlabel="Epoch", ylabel="LR", title="WarmUpCosineDecay")
ax.legend(fontsize=9); ax.grid(alpha=0.4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x:.1e}"))
plt.tight_layout()
_out = PATHS["metrics"] / "lr_schedule.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("LR schedule saved: %s", _out)

08:31:56 │ INFO     │ LR schedule saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/temporal_deepset/lr_schedule.png


## Section 7 · LR Schedule

In [ ]:
# ─── Section 7 · Temporal DeepSet Model ──────────────────────────────────────

def build_temporal_deepset(hp: dict) -> keras.Model:
    C, W, F, D = hp["MAX_CELLS"], hp["OBS_STEPS"], hp["N_FEATS"], hp["PHI_DIM"]

    inp_cells = keras.Input((C, W, F), name="cells",  dtype="float32")
    inp_mask  = keras.Input((C,),      name="mask",   dtype="float32")

    # Stage 1: shared LSTM — captures 5-second RF trends per cell
    trend = layers.TimeDistributed(
        layers.LSTM(hp["LSTM_UNITS"], return_sequences=False),
        name="td_lstm")(inp_cells)                         # (B,C,64)

    # Stage 2: Φ — non-linear per-cell embedding
    phi = trend
    for i in range(hp["PHI_LAYERS"]):
        phi = layers.TimeDistributed(
            layers.Dense(D, activation="relu"), name=f"phi_{i}")(phi)
        phi = layers.TimeDistributed(
            layers.Dropout(hp["DROPOUT"]), name=f"phi_drop_{i}")(phi)

    # Stage 3: masked global average → context z
    mask_exp = layers.Reshape((C, 1), name="mask_exp")(inp_mask)
    z        = MaskedGlobalAveragePooling(name="pool")(phi, mask_exp)   # (B,D)
    z_tiled  = layers.RepeatVector(C, name="z_tile")(z)                 # (B,C,D)

    # Stage 4: ρ — each cell sees its embedding + global context
    rho = layers.Concatenate(axis=-1, name="rho_cat")([phi, z_tiled])
    rho = layers.TimeDistributed(
        layers.Dense(hp["PHI_DIM"], activation="relu"), name="rho")(rho)
    rho = layers.TimeDistributed(
        layers.Dropout(hp["DROPOUT"]), name="rho_drop")(rho)

    # Stage 5: masked softmax — padding cells → 0 probability
    logits = layers.TimeDistributed(
        layers.Dense(1, use_bias=True), name="scorer")(rho)
    logits = layers.Reshape((C,), name="logits")(logits)
    probs  = layers.Softmax(dtype="float32", name="cell_probs")(
        layers.Add(name="pad_mask")([logits, (1.0 - inp_mask) * (-1e9)]))

    m = keras.Model([inp_cells, inp_mask], probs, name="TemporalDeepSet")
    m.compile(
        optimizer = keras.optimizers.Adam(learning_rate=lr_sched),
        loss      = LOSS_FN,
        metrics   = [
            keras.metrics.CategoricalAccuracy(name="top1_acc"),
            keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
        ],
    )
    return m


model = build_temporal_deepset(HP)
model.summary(line_length=88, expand_nested=False)
log.info("Parameters: %d", model.count_params())

Model: "TemporalDeepSet"
________________________________________________________________________________________
 Layer (type)             Output Shape              Param    Connected to               
                                                    #                                   
 cells (InputLayer)       [(None, 10, 25, 3)]       0        []                         
                                                                                        
 td_lstm (TimeDistribute  (None, 10, 64)            17408    ['cells[0][0]']            
 d)                                                                                     
                                                                                        
 phi_0 (TimeDistributed)  (None, 10, 64)            4160     ['td_lstm[0][0]']          
                                                                                        
 phi_drop_0 (TimeDistrib  (None, 10, 64)            0        ['phi_0[0][0]']         

## Section 8 · Train

In [ ]:
_out = PATHS["metrics"] / "temporal_deepset_architecture.png"

tf.keras.utils.plot_model(
    model,
    to_file=str(_out),
    show_shapes=True,
    show_layer_names=True,
    dpi=150
)

log.info("Saved: %s", _out)

if MLFLOW_OK:
    mlflow.log_artifact(str(_out))

08:31:57 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/temporal_deepset/temporal_deepset_architecture.png


In [ ]:
# ─── Training ────────────────────────────────────────────────────────────────

MONITOR  = "val_top3_acc"
CKPT_PATH = str(PATHS["models"] / "best_temporal_deepset.keras")

# 1. Close any dangling runs from previous cell executions/crashes
if MLFLOW_OK:
    while mlflow.active_run():
        mlflow.end_run()

# ── MLflow per-epoch callback ─────────────────────────────────────────────────
class MLflowEpochCB(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if MLFLOW_OK and logs:
            for k, v in logs.items():
                # This will now correctly log to the run initialized in the 'with' block
                mlflow.log_metric(k, float(v), step=epoch)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor=MONITOR, patience=12, min_delta=1e-4,
        restore_best_weights=True, mode="max", verbose=1),
    keras.callbacks.ModelCheckpoint(
        filepath=CKPT_PATH, monitor=MONITOR,
        save_best_only=True, mode="max", verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor=MONITOR, factor=0.5, patience=6,
        min_lr=1e-7, mode="max", verbose=1),
    keras.callbacks.TensorBoard(
        log_dir=str(PATHS["tb_logs"]),
        histogram_freq=0, write_graph=True, update_freq="epoch"),
    keras.callbacks.CSVLogger(
        str(PATHS["metrics"] / "training_log.csv"), append=False),
    MLflowEpochCB(),
]

log.info("Callbacks ready — monitor='%s'  ckpt='%s'", MONITOR, CKPT_PATH)
log.info("Training: %d train | %d val | batch=%d | max_ep=%d",
         len(y_tr), len(y_va), HP["BATCH_SIZE"], HP["EPOCHS"])

# 2. Setup a conditional context manager to handle with/without MLflow cleanly
import contextlib
mlflow_context = mlflow.start_run(run_name="best_temporal_deepset_v1") if MLFLOW_OK else contextlib.nullcontext()

# 3. Everything happens INSIDE the context block so the run closes automatically
with mlflow_context as run:
    if MLFLOW_OK:
        mlflow.log_params(HP)
        mlflow.log_param("loss_fn", LOSS_NAME)

    # Train model
    history = model.fit(
        ds_tr,
        validation_data = ds_va,
        epochs          = HP["EPOCHS"],
        callbacks       = callbacks,
        class_weight    = CLASS_WEIGHT,
        verbose         = 1,
    )

    # Log best metrics
    best_ep    = int(np.argmax(history.history[MONITOR])) + 1
    best_score = float(max(history.history[MONITOR]))
    log.info("Done — best %s=%.4f @ epoch %d", MONITOR, best_score, best_ep)
    
    if MLFLOW_OK:
        mlflow.log_metrics({
            "best_val_monitor.keras": best_score,
            "best_epoch.keras": float(best_ep)
        })

🏃 View run amusing-goat-106 at: http://127.0.0.1:5000/#/experiments/12/runs/6a38adbc6e6941dba800a465c245f332
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12
08:31:59 │ INFO     │ Callbacks ready — monitor='val_top3_acc'  ckpt='/home/wassimmchichi/Downloads/Handover_projects/models/best_temporal_deepset.keras'
08:31:59 │ INFO     │ Training: 57960 train | 12420 val | batch=128 | max_ep=60
Epoch 1/60


I0000 00:00:1780126326.683512   11015 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  5/453 [..............................] - ETA: 12s - loss: 0.3246 - top1_acc: 0.4156 - top3_acc: 0.6797  WARNING:tensorflow:Callback method `on_train_batch_end` is slow compared to the batch time (batch time: 0.0216s vs `on_train_batch_end` time: 0.0276s). Check your callbacks.
08:32:08 │ WARNING  │ Callback method `on_train_batch_end` is slow compared to the batch time (batch time: 0.0216s vs `on_train_batch_end` time: 0.0276s). Check your callbacks.
450/453 [============================>.] - ETA: 0s - loss: 0.1084 - top1_acc: 0.8396 - top3_acc: 0.9579
Epoch 1: val_top3_acc improved from -inf to 1.00000, saving model to /home/wassimmchichi/Downloads/Handover_projects/models/best_temporal_deepset.keras
453/453 [==============================] - 20s 25ms/step - loss: 0.1078 - top1_acc: 0.8406 - top3_acc: 0.9582 - val_loss: 0.0013 - val_top1_acc: 0.9981 - val_top3_acc: 1.0000 - lr: 2.4945e-04
Epoch 2/60
452/453 [============================>.] - ETA: 0s - loss: 0.0025 - top1_acc: 0.9922

## Section 9 · Training Curves

In [ ]:
# ─── Training Curves → metrics/ ──────────────────────────────────────────────

hist = history.history
ep   = range(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (tr_k, va_k, title, hi) in zip(axes, [
    ("loss",     "val_loss",     "Loss",                False),
    ("top1_acc", "val_top1_acc", "Top-1 Accuracy",       True),
    ("top3_acc", "val_top3_acc", "Top-3 Accuracy",       True),
]):
    ax.plot(ep, hist[tr_k], lw=2, label="train")
    ax.plot(ep, hist[va_k], lw=2, ls="--", label="val")
    fn = np.argmax if hi else np.argmin
    be, bv = fn(hist[va_k])+1, (max if hi else min)(hist[va_k])
    ax.axvline(be, color="red", ls=":", lw=1.2, alpha=0.8)
    ax.scatter([be],[bv], color="red", zorder=5, s=70,
               label=f"best @ ep {be} ({bv:.4f})")
    ax.set(title=title, xlabel="Epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.4)

fig.suptitle("Temporal DeepSet — Training History", fontsize=13, fontweight="bold")
plt.tight_layout()
_out = PATHS["metrics"] / "training_curves.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("Saved: %s", _out)
if MLFLOW_OK: mlflow.log_artifact(str(_out))

08:33:55 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/temporal_deepset/training_curves.png


## Section 10 · Evaluation (load best checkpoint)

In [ ]:
# ─── Model Evaluation — load best_model.keras from models/ ───────────────────
#
# We reload from disk (not from memory) to verify the saved artefact is
# complete and produces the same results as the in-memory model.

_BEST = str(PATHS["models"] / "best_temporal_deepset.keras")
log.info("Loading best checkpoint: %s", _BEST)

model = keras.models.load_model(
    _BEST,
    custom_objects={"MaskedGlobalAveragePooling": MaskedGlobalAveragePooling, "WarmUpCosineDecay": WarmUpCosineDecay, LOSS_FN.__name__: LOSS_FN},
)

probs_te  = model.predict(ds_te, verbose=1)   # (N, MAX_CELLS=10)
y_pred_te = probs_te.argmax(axis=1)            # integer predictions

# ── Accuracy metrics ──────────────────────────────────────────────────────────
# ALL_LABELS=[0..9] prevents the "8 classes vs 10 columns" ValueError that
# occurs because labels 8-9 (padding cells) never appear in y_te.
top1 = float((y_pred_te == y_te).mean())
top3 = float(top_k_accuracy_score(y_te, probs_te, k=3, labels=ALL_LABELS))
top5 = float(top_k_accuracy_score(y_te, probs_te, k=5, labels=ALL_LABELS))

log.info("Test → Top-1:%.4f  Top-3:%.4f  Top-5:%.4f", top1, top3, top5)
if MLFLOW_OK:
    mlflow.log_metrics({"test_top1": top1, "test_top3": top3, "test_top5": top5})

print("=" * 65)
print("  TEST RESULTS  (held-out UEs, best checkpoint)")
print("=" * 65)
print(f"  Top-1 : {top1:.4f}   ({top1*100:.2f}%)")
print(f"  Top-3 : {top3:.4f}   ({top3*100:.2f}%)")
print(f"  Top-5 : {top5:.4f}   ({top5*100:.2f}%)")
print()
print(classification_report(
    y_te,
    y_pred_te,
    labels=ALL_LABELS,  # 🔥 force all 10 classes
    target_names=[f"Cell {i}" for i in ALL_LABELS],
    digits=4,
    zero_division=0,
))

08:33:56 │ INFO     │ Loading best checkpoint: /home/wassimmchichi/Downloads/Handover_projects/models/best_temporal_deepset.keras
98/98 [==============================] - 1s 5ms/step
08:33:58 │ INFO     │ Test → Top-1:0.9982  Top-3:1.0000  Top-5:1.0000
  TEST RESULTS  (held-out UEs, best checkpoint)
  Top-1 : 0.9982   (99.82%)
  Top-3 : 1.0000   (100.00%)
  Top-5 : 1.0000   (100.00%)

              precision    recall  f1-score   support

      Cell 0     0.9984    0.9984    0.9984      1252
      Cell 1     0.9968    0.9984    0.9976      1231
      Cell 2     0.9992    0.9984    0.9988      1261
      Cell 3     0.9992    0.9975    0.9983      1204
      Cell 4     1.0000    0.9984    0.9992      1252
      Cell 5     0.9977    0.9977    0.9977      1305
      Cell 6     0.9961    0.9984    0.9973      1288
      Cell 7     1.0000    0.9984    0.9992      1255
      Cell 8     0.9968    0.9976    0.9972      1238
      Cell 9     0.9982    0.9991    0.9987      1134

    accuracy    

## Section 11 · Confusion Matrix & Per-Cell Recall

In [ ]:
# ─── Confusion Matrix & Per-Cell Recall → metrics/ ───────────────────────────

C  = HP["MAX_CELLS"]
cm = confusion_matrix(y_te, y_pred_te, labels=list(range(C)))
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
cl = [f"C{i}" for i in range(C)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[f"P-{l}" for l in cl],
            yticklabels=[f"T-{l}" for l in cl],
            linewidths=0.5, ax=axes[0], annot_kws={"size": 9})
axes[0].set(title="Confusion Matrix — Counts",
            ylabel="Actual", xlabel="Predicted")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="YlGn",
            xticklabels=[f"P-{l}" for l in cl],
            yticklabels=[f"T-{l}" for l in cl],
            linewidths=0.5, ax=axes[1], vmin=0, vmax=1,
            annot_kws={"size": 9})
axes[1].set(title="Normalised — Recall per Row",
            ylabel="Actual", xlabel="Predicted")
plt.tight_layout()
_out = PATHS["metrics"] / "confusion_matrix.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("Saved: %s", _out)
if MLFLOW_OK: mlflow.log_artifact(str(_out))

# Per-cell recall bar
per_recall = cm_norm.diagonal()
fig, ax = plt.subplots(figsize=(9, 3.5))
bars = ax.bar(range(C), per_recall,
              color=["#2196F3" if v>=0.5 else "#F44336" for v in per_recall],
              edgecolor="white", linewidth=1)
ax.axhline(top1, color="black", ls="--", lw=1.2,
           label=f"Overall Top-1 ({top1:.3f})")
for b, v in zip(bars, per_recall):
    ax.text(b.get_x()+b.get_width()/2, v+0.015,
            f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax.set(xticks=range(C), xticklabels=[f"Cell {i}" for i in range(C)],
       ylabel="Recall", ylim=(0,1.18),
       title="Per-Cell Recall — Temporal DeepSet")
ax.legend(); ax.grid(axis="y", alpha=0.4)
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
_out = PATHS["metrics"] / "per_cell_recall.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("Saved: %s", _out)
if MLFLOW_OK: mlflow.log_artifact(str(_out))

08:34:00 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/temporal_deepset/confusion_matrix.png
08:34:00 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/temporal_deepset/per_cell_recall.png


## Section 12 · Save Metadata & Final Model

In [ ]:
# ─── Save Metadata JSON → metrics/ & Final Model → models/ ───────────────────

FINAL_PATH = str(PATHS["models"] / "temporal_deepset_final.keras")
model.save(FINAL_PATH)
log.info("Final model: %s", FINAL_PATH)

meta = {
    "created"         : datetime.datetime.now().isoformat(),
    "notebook"        : "notebooks/modeling/01_temporal_deepset.ipynb",
    "best_checkpoint.keras" : _BEST,
    "final_model"     : FINAL_PATH,
    "test_top1_acc"   : round(top1, 4),
    "test_top3_acc"   : round(top3, 4),
    "test_top5_acc"   : round(top5, 4),
    "best_val_score.keras"  : round(best_score, 4),
    "best_epoch.keras"      : best_ep,
    "loss_function"   : LOSS_NAME,
    "hyperparams"     : HP,
    "paths"           : {k: str(v) for k, v in PATHS.items()},
    "timing"          : {
        "sample_rate_s": 0.2,
        "obs_s"        : HP["OBS_STEPS"] * 0.2,
        "latency_s"    : 1.0,
        "target_s"     : 1.0,
    },
}
_meta_out = PATHS["metrics"] / "temporal_deepset_metadata.json"
json.dump(meta, open(str(_meta_out), "w"), indent=2)
log.info("Metadata: %s", _meta_out)

if MLFLOW_OK:
    for p in PATHS["metrics"].glob("*.png"):
        mlflow.log_artifact(str(p))
    mlflow.log_artifact(str(_meta_out))
    mlflow.tensorflow.log_model(
        model, artifact_path="temporal_deepset_keras",
        registered_model_name="temporal_deepset")
    mlflow.end_run()
    log.info("MLflow run closed.")

print()
print("=" * 65)
print("  ARTEFACT INVENTORY")
print("=" * 65)
for lbl, d in [("models/", PATHS["models"]), ("metrics/", PATHS["metrics"])]:
    print(f"\n  {lbl}")
    for p in sorted(Path(d).iterdir()):
        if p.is_file():
            print(f"    {p.name:<42s} {p.stat().st_size/1024:>7.1f} KB")
print()
print(f"  Top-1 {top1:.4f}  |  Top-3 {top3:.4f}  |  Top-5 {top5:.4f}")
print()
print("  TensorBoard:")
print(f"    tensorboard --logdir ../../tb_logs/")
print("  MLflow UI:")
print(f"    mlflow ui --backend-store-uri file://$(pwd)/../../mlflow/mlruns")
print("  Or EC2:")
print("    ssh -N -L 5000:localhost:5000 -i key.pem ec2-user@<EC2_IP>")

08:34:01 │ INFO     │ Final model: /home/wassimmchichi/Downloads/Handover_projects/models/temporal_deepset_final.keras
08:34:01 │ INFO     │ Metadata: /home/wassimmchichi/Downloads/Handover_projects/metrics/temporal_deepset/temporal_deepset_metadata.json


2026/05/30 08:34:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 08:34:06 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: /tmp/tmpfdbvhjoo/model/data/model/assets
08:34:12 │ INFO     │ Assets written to: /tmp/tmpfdbvhjoo/model/data/model/assets


2026/05/30 08:34:21 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpfdbvhjoo/model, flavor: tensorflow). Fall back to return ['tensorflow==2.15.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
Registered model 'temporal_deepset' already exists. Creating a new version of this model...
2026/05/30 08:34:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: temporal_deepset, version 2
Created version '2' of model 'temporal_deepset'.


🏃 View run useful-boar-149 at: http://127.0.0.1:5000/#/experiments/12/runs/48d5521b8b9c41e4a3d6bf770a50ba2d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12
08:34:34 │ INFO     │ MLflow run closed.

  ARTEFACT INVENTORY

  models/
    6g_predictive_final.keras                    729.9 KB
    best_6g_predictive.keras                    9115.8 KB
    best_honet_final.keras                      3281.3 KB
    best_honet_p1.keras                         1545.1 KB
    best_honet_p2.keras                         2516.3 KB
    best_mh_transformer.keras                   3407.2 KB
    best_mtl_transformer.keras                  2096.9 KB
    best_set_transformer.keras                  2650.9 KB
    best_st_deepset.keras                       9277.2 KB
    best_strategic_deepset.keras                2404.2 KB
    best_temporal_deepset.keras                  764.1 KB
    cell_scaler.pkl                                0.6 KB
    mtl_transformer_final.keras                  518.5 KB
   

## Section 13 · Interpretation of Results

### 1. Model Architecture: Temporal DeepSet
The model uses a **Permutation Invariant** aggregator (DeepSet) to handle a variable number of neighbor cells, combined with a **Recurrent** backbone (LSTM) to capture temporal trends in signal strength (RSRP/RSRQ).

```mermaid
graph TD
    Input["Sequential Snapshots (T steps)"] --> Snap["Snapshot t"]
    Snap --> Neighs["Neighbor Set {N1, N2, ...}"]
    Neighs --> Phi["Shared Phi MLP (Feature Extractor)"]
    Phi --> Pool["Sum/Mean Pooling (Set Embedding)"]
    Pool --> LSTM["LSTM (Temporal Reasoning)"]
    LSTM --> Dense["Dense + Softmax"]
    Dense --> Output["Next Cell Prediction (C0..C9)"]
```

### 2. The "Accuracy Paradox" — Why is Cell 0 the only prediction?
During training, the model achieved **~99.7% accuracy**, yet the Confusion Matrix shows that it almost exclusively predicts **Cell 0 (T-C0)**. 

**Why this happens:**
*   **Class Imbalance:** In real-world handover datasets, UEs spend the vast majority of their time connected to a stable serving cell. Handover events (switching to C1-C9) are rare "needle-in-a-haystack" events.
*   **Path of Least Resistance:** A naive model learns that by always predicting "Stay" (Cell 0), it can achieve 99.7% accuracy with zero effort. The loss function (if not weighted) doesn't penalize the 0.3% of missed handovers enough to overcome the massive gradient from the 99.7% of correct "Stay" predictions.
*   **Failure to Trigger:** The model has failed to identify the "triggering" conditions (e.g., a neighbor's RSRP rising above the serving cell's RSRP + Hysteresis).

### 3. Visual Evidence

| Metric | Visualization | Interpretation |
| :--- | :---: | :--- |
| **Confusion Matrix** | ![Confusion Matrix](../../metrics/experiment-1/confusion_matrix.png) | Note the vertical column of predictions at **P-C0**. This indicates a high "Stay" bias. |
| **Per-Cell Recall** | ![Recall](../../metrics/experiment-1/per_cell_recall.png) | Recall for Cell 0 is near **1.0**, while recall for all neighbor cells (1-9) is near **0.0**. |
| **Training Curves** | ![Loss](../../metrics/experiment-1/training_curves.png) | A very flat loss curve after the first few epochs often indicates the model has plateaued at the majority-class baseline. |

### 4. Next Steps for Optimization
To break this bias, we must:
1.  **Use Focal Loss**: Increase the penalty for "hard" examples (the handovers) while down-weighting "easy" ones (stable serving).
2.  **Balance the Pipeline**: Use `compute_class_weight` to scale gradients based on class frequency.
3.  **Feature Engineering**: Ensure the model sees **Relative RSRP** (Serving - Neighbor) rather than absolute values, as this makes the crossover point explicit.
